1 - config

In [ ]:
from pathlib import Path
import pandas as pd 

pd.set_option("display.max_columns", None)

#Funciona tanto rodar na raíz como na pasta notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PASTA = ROOT / "data" / "raw" / "acessos_banda_larga_fixa"

arquivos = sorted(PASTA.rglob("*.csv"))
for a in arquivos:
    print(f"{a.name:55} {a.stat().st_size / 1e6:8.1f} MB")

In [15]:
arquivo = PASTA / "Acessos_Banda_Larga_Fixa_2026.csv"
print(arquivo.name)
with open(arquivo, encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())

Acessos_Banda_Larga_Fixa_2026.csv
﻿Ano;Mês;Grupo Econômico;Empresa;CNPJ;Porte da Prestadora;UF;Município;Código IBGE Município;Faixa de Velocidade;Velocidade;Tecnologia;Meio de Acesso;Tipo de Pessoa;Tipo de Produto;Acessos

2026;9;HUGHES;HUGHES;05206385000161;Pequeno Porte;MG;Itueta;3134103;2Mbps a 12Mbps;10,000000;VSAT;Satélite;Pessoa Física;INTERNET;3

2026;9;TELEFONICA;VIVO;02558157000162;Grande Porte;SP;Cosmópolis;3512803;512kbps a 2Mbps;1,000000;ADSL2;Cabo Metálico;Pessoa Física;INTERNET;4



In [17]:
amostra = pd.read_csv(arquivo, sep=";", nrows=5000, dtype=str)
print(f"Colunas da tabela: {list(amostra.columns)}")
amostra.head()

Colunas da tabela: ['Ano', 'Mês', 'Grupo Econômico', 'Empresa', 'CNPJ', 'Porte da Prestadora', 'UF', 'Município', 'Código IBGE Município', 'Faixa de Velocidade', 'Velocidade', 'Tecnologia', 'Meio de Acesso', 'Tipo de Pessoa', 'Tipo de Produto', 'Acessos']


,Ano,Mês,Grupo Econômico,Empresa,CNPJ,Porte da Prestadora,UF,Município,Código IBGE Município,Faixa de Velocidade,Velocidade,Tecnologia,Meio de Acesso,Tipo de Pessoa,Tipo de Produto,Acessos
0,2026,9,HUGHES,HUGHES,05206385000161,Pequeno Porte,MG,Itueta,3134103,2Mbps a 12Mbps,"10,000000",VSAT,Satélite,Pessoa Física,INTERNET,3
1,2026,9,TELEFONICA,VIVO,02558157000162,Grande Porte,SP,Cosmópolis,3512803,512kbps a 2Mbps,"1,000000",ADSL2,Cabo Metálico,Pessoa Física,INTERNET,4
2,2026,9,OUTROS,PB NET,05658533000189,Pequeno Porte,PB,Cuité,2505105,12Mbps a 34Mbps,"20,000000",Wi-Fi,Rádio,Pessoa Física,INTERNET,16
3,2026,9,OUTROS,G3 Telecom,13133062000113,Pequeno Porte,PI,Valença do Piauí,2211308,> 34Mbps,"600,000000",FTTH,Fibra,Pessoa Física,INTERNET,18
4,2026,9,OUTROS,WKVE TELECOM,00989304000123,Pequeno Porte,MG,Governador Valadares,3127701,> 34Mbps,"200,000000",FTTH,Fibra,Pessoa Física,INTERNET,1


In [18]:
pedacos = []
total_linhas = 0
for chunk in pd.read_csv(arquivo, sep=";", chunksize=500_000, dtype={"CNPJ": str, "Código IBGE Município":str}):
    total_linhas +=len(chunk)
    pedacos.append(chunk[chunk["UF"] == "AL"])

al = pd.concat(pedacos, ignore_index = True)
print(f"Linhas no arquivo: {total_linhas:,} | Linhas de Alagoas: {len(al):,}")

Linhas no arquivo: 6,355,655 | Linhas de Alagoas: 69,300


In [19]:
for coluna in ["Ano", "Mês", "Tecnologia", "Meio de Acesso",
               "Tipo de Pessoa", "Tipo de Produto", "Porte da Prestadora"]:
    if coluna in al.columns:
        print(f"\n--- {coluna} ---")
        print(al[coluna].value_counts().head(15))

print("\nMunicípios:", al["Município"].nunique())   # Alagoas tem 102
print("Empresas:", al["Empresa"].nunique())


--- Ano ---
Ano
2026    69300
Name: count, dtype: int64

--- Mês ---
Mês
7    8103
5    8078
6    8033
8    7957
3    7750
4    7595
2    7513
1    7189
9    7082
Name: count, dtype: int64

--- Tecnologia ---
Tecnologia
FTTH           42105
VSAT            7790
ETHERNET        5676
Wi-Fi           5137
ADSL2           1787
HDSL            1723
FTTB            1374
ADSL1           1280
HFC              970
DWDM             630
TDMA             330
Cable Modem      182
FWA              172
FR                82
OFDMA/TDD         53
Name: count, dtype: int64

--- Meio de Acesso ---
Meio de Acesso
Fibra            48409
Satélite          8129
Cabo Metálico     6322
Rádio             5405
Cabo Coaxial      1035
Name: count, dtype: int64

--- Tipo de Pessoa ---
Tipo de Pessoa
Pessoa Física      35460
Pessoa Jurídica    33840
Name: count, dtype: int64

--- Tipo de Produto ---
Tipo de Produto
INTERNET          59146
LINHA_DEDICADA     6269
M2M                3315
OUTROS              570
Name: 

In [20]:
aloo = al[al["CNPJ"] == "05680391000156"]
print(f"Linhas da Aloo: {len(aloo):,}")
print(aloo.groupby(["Tipo de Produto", "Tipo de Pessoa"])["Acessos"].sum())

Linhas da Aloo: 0
Series([], Name: Acessos, dtype: int64)


In [21]:
padrao = "ALOO|FSF"
mascara = (
    al["CNPJ"].str.startswith("05680391", na=False)
    | al["Empresa"].str.contains(padrao, case=False, na=False)
    | al["Grupo Econômico"].str.contains(padrao, case=False, na=False)
)
al[mascara][["Grupo Econômico", "Empresa", "CNPJ"]].drop_duplicates()

,Grupo Econômico,Empresa,CNPJ


In [22]:
COLS = ["Ano", "Mês", "Grupo Econômico", "Empresa", "CNPJ", "UF",
        "Tipo de Pessoa", "Tipo de Produto", "Acessos"]

def buscar_aloo(nome_arquivo):
    achados = []
    for chunk in pd.read_csv(PASTA / nome_arquivo, sep=";", chunksize=500_000,
                             usecols=COLS, dtype={"CNPJ": str}):
        m = (chunk["CNPJ"].str.startswith("05680391", na=False)
             | chunk["Empresa"].str.contains(padrao, case=False, na=False)
             | chunk["Grupo Econômico"].str.contains(padrao, case=False, na=False))
        achados.append(chunk[m])
    return pd.concat(achados, ignore_index=True)

for nome in ["Acessos_Banda_Larga_Fixa_2026.csv",
             "Acessos_Banda_Larga_Fixa_2023.csv",
             "Acessos_Banda_Larga_Fixa_2021.csv"]:
    res = buscar_aloo(nome)
    print(f"\n=== {nome}: {len(res)} linhas ===")
    if len(res):
        print(res.groupby(["Ano", "UF", "Empresa", "CNPJ", "Tipo de Produto"])["Acessos"].sum())


=== Acessos_Banda_Larga_Fixa_2026.csv: 0 linhas ===

=== Acessos_Banda_Larga_Fixa_2023.csv: 1198 linhas ===
Ano   UF  Empresa       CNPJ            Tipo de Produto
2023  AL  ALOO TELECOM  05680391000156  INTERNET           1363
                                        LINHA_DEDICADA     1455
                                        OUTROS             1024
      BA  ALOO TELECOM  05680391000156  INTERNET             41
                                        LINHA_DEDICADA       82
                                        OUTROS               25
      CE  ALOO TELECOM  05680391000156  INTERNET             60
                                        LINHA_DEDICADA      131
                                        OUTROS               52
      DF  ALOO TELECOM  05680391000156  LINHA_DEDICADA        2
      PB  ALOO TELECOM  05680391000156  INTERNET             27
                                        LINHA_DEDICADA       51
                                        OUTROS                4
   

In [23]:
setembro = al[al["Mês"] == al["Mês"].max()]
print("Total de acessos em AL (set/2026):", f"{setembro['Acessos'].sum():,}")
(setembro.groupby(["Grupo Econômico", "Empresa"])["Acessos"]
         .sum().sort_values(ascending=False).head(15))

Total de acessos em AL (set/2026): 387,165


Grupo Econômico   Empresa                                   
BRISANET          BRISANET                                      116856
TELECOM AMERICAS  CLARO                                          86617
TELEFONICA        VIVO                                           21543
OUTROS            Dtel                                           13356
OI                OI                                              5991
OUTROS            TELECOM INTERNET                                5788
                  G S Net                                         3850
                  REELUNET                                        3679
                  R E COMERCIO E SERVICOS DE INTERNET LTDA        3480
                  SPEEDY TELECOM                                  3353
                  Inova +                                         2928
                  STARLINK BRAZIL SERVICOS DE INTERNET LTDA.      2859
                  MARBRIELE                                       2831
                

In [24]:
for nome in ["Acessos_Banda_Larga_Fixa_2022.csv",
             "Acessos_Banda_Larga_Fixa_2023.csv",
             "Acessos_Banda_Larga_Fixa_2024.csv",
             "Acessos_Banda_Larga_Fixa_2025.csv"]:
    res = buscar_aloo(nome)
    print(f"\n=== {nome}: {len(res)} linhas ===")
    if len(res):
        print(res.groupby(["Ano", "Mês"])["Acessos"].sum())


=== Acessos_Banda_Larga_Fixa_2022.csv: 11019 linhas ===
Ano   Mês
2022  3      4617
      4      4785
      5      4974
      6      5164
      7      5176
      8      5125
      9      5200
      10     5065
      11     5185
      12     5335
Name: Acessos, dtype: int64

=== Acessos_Banda_Larga_Fixa_2023.csv: 1198 linhas ===
Ano   Mês
2023  3      5468
Name: Acessos, dtype: int64

=== Acessos_Banda_Larga_Fixa_2024.csv: 0 linhas ===

=== Acessos_Banda_Larga_Fixa_2025.csv: 6327 linhas ===
Ano   Mês
2025  6      3743
      7      3808
      8      3959
      9      4019
Name: Acessos, dtype: int64


In [25]:
for a in sorted(PASTA.glob("Acessos_Banda_Larga_Fixa_*.csv")):
    if "Colunas" in a.name or "Total" in a.name:
        continue
    cols = pd.read_csv(a, sep=";", nrows=1).columns.tolist()
    print(f"{a.name:45} {len(cols)} colunas")

Acessos_Banda_Larga_Fixa_2007-2010.csv        13 colunas
Acessos_Banda_Larga_Fixa_2011-2012.csv        13 colunas
Acessos_Banda_Larga_Fixa_2013-2014.csv        13 colunas
Acessos_Banda_Larga_Fixa_2015-2016.csv        13 colunas
Acessos_Banda_Larga_Fixa_2017-2018.csv        13 colunas
Acessos_Banda_Larga_Fixa_2019-2020.csv        13 colunas
Acessos_Banda_Larga_Fixa_2021.csv             16 colunas
Acessos_Banda_Larga_Fixa_2022.csv             16 colunas
Acessos_Banda_Larga_Fixa_2023.csv             16 colunas
Acessos_Banda_Larga_Fixa_2024.csv             16 colunas
Acessos_Banda_Larga_Fixa_2025.csv             16 colunas
Acessos_Banda_Larga_Fixa_2026.csv             16 colunas
